In [11]:
# Graph Builder

def build_graph(row):
    s = CifParser(os.path.join(CIF_DIR, row['cif'])).get_structures()[0]

    # Node features
    x = torch.tensor(node_scaler.transform(
        [get_node_features(site) for site in s]), dtype=torch.float)

    pos = torch.tensor(s.cart_coords, dtype=torch.float)

    # Edges
    edge_index = radius_graph(pos, r=CUTOFF, loop=False)
    src, dst = edge_index

    dist = torch.norm(pos[src] - pos[dst], dim=1, keepdim=True)
    edge_attr = gaussian_expand(dist)

    # 🔥 FIX: normalize edge features
    edge_attr = (edge_attr - edge_attr.mean(0)) / (edge_attr.std(0) + 1e-6)

    # Global features
    g = [
        row['spacegroup_number'], row['number of atoms'],
        row.get('Band Gap', 0),
        row['a'], row['b'], row['c'],
        row['alpha'], row['beta'], row['gamma'],
        row['Z'], row['electronegativity'],
        s.volume, s.density, len(s.composition.elements)
    ]

    u = torch.tensor(global_scaler.transform([g]), dtype=torch.float)

    y = torch.tensor([row['label']], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                u=u, y=y, pos=pos)

In [13]:
# Dataset
class CrystalDataset(Dataset):
    def __init__(self, df):
        super().__init__() # Initialize the base Dataset class
        self.df = df

    def len(self):
        return len(self.df)

    def get(self, idx):
        return build_graph(self.df.iloc[idx])

train_loader = DataLoader(CrystalDataset(train_df), batch_size=32, shuffle=True)
val_loader   = DataLoader(CrystalDataset(val_df), batch_size=32)
test_loader  = DataLoader(CrystalDataset(test_df), batch_size=32)

In [14]:
# CharlesCGCNN Model

class CharlesCGCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.emb = nn.Linear(8, 128)

        self.conv1 = CGConv(128, dim=40)
        self.conv2 = CGConv(128, dim=40)
        self.conv3 = CGConv(128, dim=40)

        self.fc1 = nn.Linear(128 + 14, 128)
        self.fc2 = nn.Linear(128, 2)

        self.dropout = nn.Dropout(0.2)

    def forward(self, data):
        x, ei, ea, batch, u = data.x, data.edge_index, data.edge_attr, data.batch, data.u
        if u.dim() == 3: u = u.squeeze(1)

        x = F.relu(self.emb(x))
        x = F.relu(self.conv1(x, ei, ea))
        x = F.relu(self.conv2(x, ei, ea))
        x = F.relu(self.conv3(x, ei, ea))

        x = global_add_pool(x, batch)

        x = torch.cat([x, u], dim=1)

        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

model = CharlesCGCNN().to(device)

In [16]:
# Training Setup

# Class weights
counts = train_df['label'].value_counts()
w0 = len(train_df) / (2 * counts[0])
w1 = len(train_df) / (2 * counts[1])

criterion = nn.CrossEntropyLoss(weight=torch.tensor([w0, w1], dtype=torch.float32).to(device))

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

best_auc = 0

for epoch in range(1, 101):
    model.train()
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch), batch.y)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    preds, probs, labels = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch)
            preds += out.argmax(1).cpu().tolist()
            probs += torch.softmax(out,1)[:,1].cpu().tolist()
            labels += batch.y.cpu().tolist()

    auc = roc_auc_score(labels, probs)
    scheduler.step(1 - auc)

    print(f"Epoch {epoch} | Val AUC: {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), BASE_DIR + "/best_cgcnn.pth")

Epoch 1 | Val AUC: 0.8045
Epoch 2 | Val AUC: 0.8170
Epoch 3 | Val AUC: 0.8360
Epoch 4 | Val AUC: 0.8203
Epoch 5 | Val AUC: 0.8460
Epoch 6 | Val AUC: 0.8542
Epoch 7 | Val AUC: 0.8500
Epoch 8 | Val AUC: 0.8667
Epoch 9 | Val AUC: 0.8675
Epoch 10 | Val AUC: 0.8642
Epoch 11 | Val AUC: 0.8660
Epoch 12 | Val AUC: 0.8722
Epoch 13 | Val AUC: 0.8635
Epoch 14 | Val AUC: 0.8658
Epoch 15 | Val AUC: 0.8809
Epoch 16 | Val AUC: 0.8849
Epoch 17 | Val AUC: 0.8696
Epoch 18 | Val AUC: 0.8850
Epoch 19 | Val AUC: 0.8824
Epoch 20 | Val AUC: 0.8729
Epoch 21 | Val AUC: 0.8730
Epoch 22 | Val AUC: 0.8763
Epoch 23 | Val AUC: 0.8749
Epoch 24 | Val AUC: 0.8748
Epoch 25 | Val AUC: 0.8800
Epoch 26 | Val AUC: 0.8845
Epoch 27 | Val AUC: 0.8836
Epoch 28 | Val AUC: 0.8848
Epoch 29 | Val AUC: 0.8874
Epoch 30 | Val AUC: 0.8862
Epoch 31 | Val AUC: 0.8876
Epoch 32 | Val AUC: 0.8882
Epoch 33 | Val AUC: 0.8897
Epoch 34 | Val AUC: 0.8887
Epoch 35 | Val AUC: 0.8884
Epoch 36 | Val AUC: 0.8896
Epoch 37 | Val AUC: 0.8899
Epoch 38 |

In [17]:
# Test Evaluation

model.load_state_dict(torch.load(BASE_DIR + "/best_cgcnn.pth"))
model.eval()

preds, probs, labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch)
        preds += out.argmax(1).cpu().tolist()
        probs += torch.softmax(out,1)[:,1].cpu().tolist()
        labels += batch.y.cpu().tolist()

print("Accuracy:", accuracy_score(labels, preds))
print("F1:", f1_score(labels, preds))
print("ROC-AUC:", roc_auc_score(labels, probs))

Accuracy: 0.8180952380952381
F1: 0.828082808280828
ROC-AUC: 0.8927279390686855
